# 07 -- Weather / Air-Density Analysis (Contact Luck v0.5)

**Contact Luck Prototype v0.5**

Does incorporating actual game-time weather (temperature, humidity, pressure, wind, roof status) and derived air density improve out-of-sample probability quality over the currently-selected production baseline?

Controlled variants, same 2021-2023 training rows, same untouched 2024 validation rows (2025 never touched):

- `selected_production_baseline` -- the current default model (as of this notebook, `baseline_v02`: no venue, no geometry -- see README.md "Current status").
- `weather_basic_v05_candidate` -- baseline + temperature/humidity/air density/roof-indoor status.
- `weather_vector_v05_candidate` -- baseline + air density (and its deviation from a fixed reference atmosphere) + following/head/crosswind (relative to each play's own spray direction) + roof-adjusted conditions.
- `geometry_plus_weather_v05_candidate` -- baseline + park geometry (Version 0.4, itself NOT adopted) + the weather-vector feature set.

**Adoption rule: this notebook does NOT automatically prefer a weather-aware candidate.** See the recommendation section near the end -- it checks 8 of the task's 10 criteria automatically; "learned effects are physically plausible" is a judgment call, never automated.

**Contact Luck v0.5 remains a contact-only research prototype.** It does not yet include exact defensive positioning, defensive execution, batter-runner advancement, or full multi-factor Shapley attribution.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from mlb_luck_score.config import CLASS_ORDER, PROCESSED_DATA_DIR, RAW_DATA_DIR, TRAIN_SEASONS, VALIDATION_SEASONS
from mlb_luck_score.data.venue_environment import VENUE_ENVIRONMENTS, get_venue_environment
from mlb_luck_score.data.weather_physics import (
    DEFAULT_REFERENCE_AIR_DENSITY_KG_M3,
    moist_air_density_kg_m3,
    parse_wind_text,
    pressure_at_elevation_hpa,
    wind_relative_components,
)
from mlb_luck_score.data.join_weather_features import join_weather_features, build_weather_join_report
from mlb_luck_score.features.build_contact_features import (
    WEATHER_VECTOR_NUMERIC_FEATURES,
    generate_standardized_environment_rows,
)
from mlb_luck_score.models.compare_weather_aware import (
    VARIANT_SELECTED_PRODUCTION_BASELINE,
    VARIANT_WEATHER_BASIC_V05_CANDIDATE,
    VARIANT_WEATHER_VECTOR_V05_CANDIDATE,
    VARIANT_GEOMETRY_PLUS_WEATHER_V05_CANDIDATE,
    _prepare_weather_columns,
    check_standardized_predictions_stable,
    compute_paired_bootstrap,
    recommend_weather_adoption,
    run_weather_aware_comparison,
)
from mlb_luck_score.models.train_contact_model import predict_proba_ordered
from mlb_luck_score.scoring.weather_attribution import compute_weather_attribution

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 60)

GAME_WEATHER_PATH = PROCESSED_DATA_DIR / "game_weather.parquet"
WEATHER_JOINED_PATH = PROCESSED_DATA_DIR / "cleaned_development_data_with_weather.parquet"
GEOMETRY_JOINED_PATH = PROCESSED_DATA_DIR / "cleaned_development_data_with_geometry.parquet"

if WEATHER_JOINED_PATH.exists():
    df = pd.read_parquet(WEATHER_JOINED_PATH)
    print(f"Loaded pre-joined weather data: {len(df)} rows from {WEATHER_JOINED_PATH}")
elif GEOMETRY_JOINED_PATH.exists() and GAME_WEATHER_PATH.exists():
    geometry_df = pd.read_parquet(GEOMETRY_JOINED_PATH)
    game_weather_df = pd.read_parquet(GAME_WEATHER_PATH)
    df = join_weather_features(geometry_df, game_weather_df)
    print(f"Joined weather on the fly: {len(df)} rows")
else:
    df = None
    print(
        "No weather-joined data found. Run `make download-weather-data && "
        "make build-game-weather && make join-weather-features` first, then re-run this notebook."
    )

game_weather_df = pd.read_parquet(GAME_WEATHER_PATH) if GAME_WEATHER_PATH.exists() else None

Loaded pre-joined weather data: 494173 rows from /Users/arihantaneja/Downloads/TrueLuckMLBStat/data/processed/cleaned_development_data_with_weather.parquet


## 1. Source and provenance overview

Two independent, complementary sources -- see `mlb_luck_score.data.download_historical_weather` module docstring:

1. **MLB schedule weather** (`schedule?hydrate=weather`): official per-game temperature, wind (already expressed relative to the park, e.g. `"8 mph, Out To CF"`), and condition/roof text (e.g. `"Roof Closed"`).
2. **Iowa Environmental Mesonet ASOS archive** (public, no API key): routine hourly humidity/pressure observations from each venue's assigned station -- the ONLY source of humidity/pressure in this repository.

In [2]:
print(f"Venues with reviewed environment reference data: {len(VENUE_ENVIRONMENTS)}")
print(f"Unique weather stations assigned: {len({v.weather_station_id for v in VENUE_ENVIRONMENTS})}")
print(f"\nSchedule-weather source: MLB Stats API /schedule?hydrate=weather")
print(f"Station-observation source: https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py (report_type=3, routine hourly)")
review_statuses = {v.review_status for v in VENUE_ENVIRONMENTS}
print(f"Venue-environment review status(es): {review_statuses} (agent-sourced, pending human review -- see module docstring)")

Venues with reviewed environment reference data: 30
Unique weather stations assigned: 29

Schedule-weather source: MLB Stats API /schedule?hydrate=weather
Station-observation source: https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py (report_type=3, routine hourly)
Venue-environment review status(es): {'agent_sourced_pending_human_review'} (agent-sourced, pending human review -- see module docstring)


## 2. Weather-station and venue mapping

In [3]:
rows = []
for v in sorted(VENUE_ENVIRONMENTS, key=lambda v: v.venue_name):
    rows.append({
        "venue_name": v.venue_name,
        "weather_station_id": v.weather_station_id,
        "weather_station_name": v.weather_station_name,
        "station_distance_km": round(v.station_distance_km(), 1),
        "elevation_m": v.elevation_m,
        "elevation_source": v.elevation_source,
        "timezone": v.timezone,
    })
station_df = pd.DataFrame(rows)
display(station_df)
print(f"\nMax station distance: {station_df['station_distance_km'].max():.1f} km")
print(f"Mean station distance: {station_df['station_distance_km'].mean():.1f} km")

,venue_name,weather_station_id,weather_station_name,station_distance_km,elevation_m,elevation_source,timezone
0,American Family Field,MKE,Milwaukee Mitchell International Airport,10.8,204.000,nearest_weather_station_elevation_proxy,America/Chicago
1,Angel Stadium,SNA,"John Wayne Airport, Santa Ana CA",13.9,16.000,nearest_weather_station_elevation_proxy,America/Los_Angeles
2,Busch Stadium,STL,St. Louis Lambert International Airport,21.3,171.000,nearest_weather_station_elevation_proxy,America/Chicago
3,Chase Field,PHX,Phoenix Sky Harbor International Airport,5.3,337.000,nearest_weather_station_elevation_proxy,America/Phoenix
4,Citi Field,LGA,"LaGuardia Airport, New York NY",3.8,9.000,nearest_weather_station_elevation_proxy,America/New_York
5,Citizens Bank Park,PHL,Philadelphia International Airport,6.3,2.000,nearest_weather_station_elevation_proxy,America/New_York
6,Comerica Park,DET,"Coleman A. Young International Airport, Detroi...",8.4,190.000,nearest_weather_station_elevation_proxy,America/Detroit
7,Coors Field,BKF,"Buckley Space Force Base, Aurora CO",21.6,1609.344,stadium_specific_published_value,America/Denver
8,Dodger Stadium,BUR,Hollywood Burbank Airport,17.8,236.000,nearest_weather_station_elevation_proxy,America/Los_Angeles
9,Fenway Park,BOS,Boston Logan International Airport,7.4,9.000,nearest_weather_station_elevation_proxy,America/New_York



Max station distance: 22.9 km
Mean station distance: 11.3 km


## 3. Weather coverage by season and venue

In [4]:
if df is not None:
    report = build_weather_join_report(df)
    print(f"Total rows: {report.total_rows}")
    print(f"Rows with ANY weather data: {report.rows_with_weather_data} ({report.coverage_rate:.2%})")
    print(f"Rows with EFFECTIVE (actionable) weather: {report.rows_with_effective_weather} ({report.effective_coverage_rate:.2%})")
    print(f"\nCounts by weather_status:")
    for status, count in sorted(report.counts_by_status.items(), key=lambda kv: -kv[1]):
        print(f"  {status:40s} {count:>8d}")
    print(f"\nCoverage by season:")
    for season, n in sorted(report.counts_by_season.items()):
        total = int((df["season"].astype(str) == str(season)).sum())
        print(f"  {season}: {n}/{total} ({n/total:.2%})")
else:
    report = None
    print("Skipped -- no data loaded.")

Total rows: 494173
Rows with ANY weather data: 491026 (99.36%)
Rows with EFFECTIVE (actionable) weather: 401567 (81.26%)

Counts by weather_status:
  ok                                         401567
  effective_conditions_unavailable_indoor     86865
  venue_environment_unavailable                3147
  no_observation_within_window                 2594

Coverage by season:
  2021: 98333/121699 (80.80%)
  2022: 102692/124261 (82.64%)


  2023: 100818/124233 (81.15%)
  2024: 99724/123980 (80.44%)


In [5]:
if df is not None:
    venue_coverage = df.groupby("venue_name")["has_effective_weather"].agg(["sum", "size"])
    venue_coverage["rate"] = venue_coverage["sum"] / venue_coverage["size"]
    display(venue_coverage.sort_values("rate").rename(columns={"sum": "effective_rows", "size": "total_rows"}))
else:
    print("Skipped -- no data loaded.")

,effective_rows,total_rows,rate
venue_name,,,
Tropicana Field,0,15744,0.000000
BB&T Ballpark,0,43,0.000000
TD Ballpark,0,1160,0.000000
Sahlen Field,0,1238,0.000000
Rickwood Field,0,47,0.000000
Muncy Bank Ballpark,0,95,0.000000
"MLB Field of Dreams (Dyersville, Iowa)",0,87,0.000000
London Stadium,0,217,0.000000
Journey Bank Ballpark,0,48,0.000000


## 4. Observation-time-offset distribution

In [6]:
if game_weather_df is not None:
    offsets = game_weather_df["weather_time_offset_minutes"].dropna()
    print(offsets.describe())
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(offsets.clip(-60, 200), bins=60)
    ax.set_xlabel("weather_time_offset_minutes (positive = observation before game start)")
    ax.set_ylabel("count (games)")
    ax.set_title("Observation-to-game-start time offset")
    plt.tight_layout()
    plt.show()
    print(f"\nGood match (<=60 min): {(offsets.abs() <= 60).mean():.2%}")
    print(f"Fair match (<=180 min): {(offsets.abs() <= 180).mean():.2%}")
else:
    print("Skipped -- no game-weather table loaded.")

count    9595.000000
mean       25.981761
std        14.855098
min       -43.000000
25%        16.000000
50%        17.000000
75%        42.000000
max       137.000000
Name: weather_time_offset_minutes, dtype: float64



Good match (<=60 min): 99.80%
Fair match (<=180 min): 100.00%


/var/folders/k_/c1x6qgmx0b56khm3k49fgfq80000gn/T/ipykernel_97699/4219122266.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Roof-status coverage

In [7]:
if game_weather_df is not None:
    counts = game_weather_df["roof_status"].value_counts(dropna=False)
    display(counts.to_frame("games"))
    fig, ax = plt.subplots(figsize=(7, 5))
    counts.sort_values().plot(kind="barh", ax=ax)
    ax.set_xlabel("games")
    ax.set_title("Roof status (2021-2024)")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped -- no game-weather table loaded.")

,games
roof_status,
outdoor_open_air,7170
retractable_roof_closed,1395
retractable_roof_open,827
fixed_indoor,324
roof_status_unknown,2


/var/folders/k_/c1x6qgmx0b56khm3k49fgfq80000gn/T/ipykernel_97699/2809656954.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Air-density formula and units

Moist-air density (`mlb_luck_score.data.weather_physics.moist_air_density_kg_m3`) uses the ideal gas law applied separately to dry-air and water-vapor partial pressures:

```
e   = saturation_vapor_pressure_hpa(T) * (RH / 100)   -- Arden Buck equation
p_d = P - e
rho = (p_d * 100) / (R_d * T_K) + (e * 100) / (R_v * T_K)
```

Station pressure is estimated from each station's sea-level-reduced pressure (`mslp`) via the hypsometric equation, evaluated at the VENUE's own elevation (not the station's) -- see `pressure_at_elevation_hpa`. Reference atmosphere: ICAO ISA sea level (15 deg C, 1013.25 hPa, 0% RH, `DEFAULT_REFERENCE_AIR_DENSITY_KG_M3` = 1.225 kg/m^3).

In [8]:
print(f"Reference air density (ISA sea level): {DEFAULT_REFERENCE_AIR_DENSITY_KG_M3} kg/m^3")
coors_pressure = pressure_at_elevation_hpa(1013.0, 1609.344, 25.0)
coors_density = moist_air_density_kg_m3(25.0, coors_pressure, 30.0)
sea_level_density = moist_air_density_kg_m3(25.0, 1013.0, 30.0)
print(f"Example -- Coors Field (elevation 1609m, 25C, 30% RH): estimated pressure={coors_pressure:.1f} hPa, "
      f"density={coors_density:.4f} kg/m^3 ({(1 - coors_density/sea_level_density):.1%} less dense than sea level at the same temp/humidity)")

Reference air density (ISA sea level): 1.225 kg/m^3
Example -- Coors Field (elevation 1609m, 25C, 30% RH): estimated pressure=842.4 hPa, density=0.9801 kg/m^3 (16.9% less dense than sea level at the same temp/humidity)


## 7. Air-density distribution

In [9]:
if df is not None:
    density = df.loc[df["has_effective_weather"], "air_density_kg_m3"]
    print(density.describe())
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(density, bins=80)
    ax.axvline(DEFAULT_REFERENCE_AIR_DENSITY_KG_M3, color="red", linestyle="--", label="ISA reference")
    ax.set_xlabel("air_density_kg_m3")
    ax.set_ylabel("count")
    ax.legend()
    plt.tight_layout()
    plt.show()

    by_venue_density = df[df["has_effective_weather"]].groupby("venue_name")["air_density_kg_m3"].mean().sort_values()
    print("\nLowest mean air density (thinnest air):")
    display(by_venue_density.head(5))
    print("\nHighest mean air density (densest air):")
    display(by_venue_density.tail(5))
else:
    print("Skipped -- no data loaded.")

count    399866.000000
mean          1.161749
std           0.050580
min           0.945719
25%           1.142597
50%           1.165305
75%           1.192019
max           1.287813
Name: air_density_kg_m3, dtype: float64


/var/folders/k_/c1x6qgmx0b56khm3k49fgfq80000gn/T/ipykernel_97699/131150341.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Lowest mean air density (thinnest air):


venue_name
Coors Field         0.981916
Chase Field         1.106614
PNC Park            1.135753
Truist Park         1.141025
Kauffman Stadium    1.144642
Name: air_density_kg_m3, dtype: float64


Highest mean air density (densest air):


venue_name
Petco Park          1.192502
Fenway Park         1.192955
Oakland Coliseum    1.198650
T-Mobile Park       1.207301
Oracle Park         1.209925
Name: air_density_kg_m3, dtype: float64

## 8. Temperature, pressure, humidity, and wind distributions

In [10]:
if df is not None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    avail = df["has_effective_weather"]
    axes[0,0].hist(df.loc[avail, "temperature_c"], bins=60); axes[0,0].set_title("Temperature (C)")
    axes[0,1].hist(df.loc[avail, "pressure_hpa"], bins=60); axes[0,1].set_title("Pressure (hPa)")
    axes[1,0].hist(df.loc[avail, "humidity_pct"], bins=60); axes[1,0].set_title("Relative humidity (%)")
    axes[1,1].hist(df.loc[avail, "wind_speed_mps"], bins=60); axes[1,1].set_title("Wind speed (m/s)")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped -- no data loaded.")

/var/folders/k_/c1x6qgmx0b56khm3k49fgfq80000gn/T/ipykernel_97699/2748208873.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Stadium-orientation coverage

MLB's own wind text is already reported RELATIVE to each park's own orientation (e.g. `"Out To CF"`, `"L To R"`) -- this repository never needed to independently source true-north field-orientation angles per park (a documented gap the task allowed deferring). Every play's following/head/crosswind decomposition (`mlb_luck_score.data.weather_physics.wind_relative_components`) is well-defined whenever `effective_wind_movement_bearing_degrees` and that play's own `spray_angle_approx` are both available.

In [11]:
if df is not None:
    has_bearing = df["has_effective_weather"] & df["following_wind_mps"].notna()
    print(f"Rows with a usable wind-relative decomposition: {has_bearing.sum()} / {df['has_effective_weather'].sum()} "
          f"({has_bearing.sum() / max(df['has_effective_weather'].sum(), 1):.2%} of effective-weather rows)")
    calm_or_unparseable = df["has_effective_weather"] & df["following_wind_mps"].isna()
    print(f"Effective-weather rows WITHOUT a usable wind vector (calm/unparseable direction): {calm_or_unparseable.sum()}")
else:
    print("Skipped -- no data loaded.")

Rows with a usable wind-relative decomposition: 377563 / 401567 (94.02% of effective-weather rows)
Effective-weather rows WITHOUT a usable wind vector (calm/unparseable direction): 24004


## 10. Following/headwind/crosswind examples

In [12]:
examples = [
    ("10 mph, Out To CF", 0.0, "Ball to CF, wind blowing straight out to CF"),
    ("10 mph, In From CF", 0.0, "Ball to CF, wind blowing straight in from CF"),
    ("8 mph, L To R", 0.0, "Ball to CF, pure L-to-R crosswind"),
    ("8 mph, Out To RF", 45.0, "Ball to RF line, wind blowing out to RF (pure following)"),
    ("8 mph, Out To CF", 45.0, "Ball to RF line, wind blowing out to CF (mixed)"),
]
rows = []
for wind_text, spray_angle, description in examples:
    wind = parse_wind_text(wind_text)
    comp = wind_relative_components(wind, spray_angle)
    rows.append({
        "description": description, "wind_text": wind_text, "spray_angle": spray_angle,
        "following_wind_mps": round(comp.following_wind_mps, 2),
        "headwind_mps": round(comp.headwind_mps, 2),
        "crosswind_mps": round(comp.crosswind_mps, 2),
    })
display(pd.DataFrame(rows))

,description,wind_text,spray_angle,following_wind_mps,headwind_mps,crosswind_mps
0,"Ball to CF, wind blowing straight out to CF","10 mph, Out To CF",0.0,4.47,-4.47,0.00
1,"Ball to CF, wind blowing straight in from CF","10 mph, In From CF",0.0,-4.47,4.47,-0.00
2,"Ball to CF, pure L-to-R crosswind","8 mph, L To R",0.0,0.00,-0.00,3.58
3,"Ball to RF line, wind blowing out to RF (pure ...","8 mph, Out To RF",45.0,3.58,-3.58,0.00
4,"Ball to RF line, wind blowing out to CF (mixed)","8 mph, Out To CF",45.0,2.53,-2.53,-2.53


## 11. Four-way model comparison

In [13]:
if df is not None:
    include_geometry = "wall_distance_in_spray_direction" in df.columns
    comparison, trained_models, proba_by_variant = run_weather_aware_comparison(df, include_geometry=include_geometry)
    summary_rows = []
    for variant, s in comparison.items():
        summary_rows.append({
            "variant": variant,
            "multiclass_log_loss": s["multiclass_log_loss"],
            "expected_calibration_error": s["expected_calibration_error"],
            "home_run_ece": s["home_run_ece"],
            "argmax_accuracy (secondary)": s["argmax_accuracy_secondary"],
            "weather_coverage_rate": s.get("weather_coverage_rate"),
            "n": s["sample_count"],
        })
    display(pd.DataFrame(summary_rows).set_index("variant"))
else:
    comparison, trained_models, proba_by_variant = None, None, None
    print("Skipped -- no data loaded.")

6 of 41 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 36 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 35 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 36 calibration bin(s) have fewer than 20 samples and are marked unreliable.


2 of 35 calibration bin(s) have fewer than 20 samples and are marked unreliable.


...

14 of 42 calibration bin(s) have fewer than 20 samples and are marked unreliable.


11 of 41 calibration bin(s) have fewer than 20 samples and are marked unreliable.


2 of 35 calibration bin(s) have fewer than 20 samples and are marked unreliable.


4 of 45 calibration bin(s) have fewer than 20 samples and are marked unreliable.


,multiclass_log_loss,expected_calibration_error,home_run_ece,argmax_accuracy (secondary),weather_coverage_rate,n
variant,,,,,,
selected_production_baseline,0.670321,0.014413,0.002706,0.751457,0.804204,122132
weather_basic_v05_candidate,0.670015,0.014500,0.003401,0.751777,0.804204,122132
weather_vector_v05_candidate,0.670038,0.014391,0.003409,0.751572,0.804204,122132
geometry_plus_weather_v05_candidate,0.562149,0.016042,0.001047,0.778355,0.804204,122132


## 12. Calibration by outcome class

In [14]:
if comparison is not None:
    class_rows = []
    for variant in comparison:
        row = {"variant": variant}
        row.update(comparison[variant]["expected_calibration_error_by_class"])
        class_rows.append(row)
    display(pd.DataFrame(class_rows).set_index("variant")[list(CLASS_ORDER)])
else:
    print("Skipped -- no comparison available.")

,out,single,double,triple,home_run
variant,,,,,
selected_production_baseline,0.031353,0.033749,0.004039,0.000218,0.002706
weather_basic_v05_candidate,0.030671,0.034119,0.004067,0.000245,0.003401
weather_vector_v05_candidate,0.030517,0.033809,0.004012,0.000205,0.003409
geometry_plus_weather_v05_candidate,0.035890,0.040315,0.002633,0.000327,0.001047


## 13. Calibration by venue

In [15]:
if comparison is not None:
    base_venue = pd.DataFrame(comparison[VARIANT_SELECTED_PRODUCTION_BASELINE]["calibration_by_venue"])
    candidates = [v for v in comparison if v != VARIANT_SELECTED_PRODUCTION_BASELINE]
    for candidate in candidates:
        cand_venue = pd.DataFrame(comparison[candidate]["calibration_by_venue"])
        by_venue = base_venue.merge(cand_venue, on="venue_id", suffixes=("_baseline", "_candidate"))
        by_venue["ece_delta"] = by_venue["ece_overall_candidate"] - by_venue["ece_overall_baseline"]
        by_venue = by_venue.sort_values("ece_delta")
        print(f"\n=== {candidate} vs baseline (largest regressions last) ===")
        display(by_venue[["venue_id", "sample_count_baseline", "ece_overall_baseline", "ece_overall_candidate", "ece_delta", "reliable_baseline"]].tail(8))
else:
    print("Skipped -- no comparison available.")


=== weather_basic_v05_candidate vs baseline (largest regressions last) ===


,venue_id,sample_count_baseline,ece_overall_baseline,ece_overall_candidate,ece_delta,reliable_baseline
18,1,4010,0.020433,0.020840,0.000406,True
23,2394,3981,0.023499,0.023918,0.000418,True
12,10,4110,0.014923,0.015463,0.000540,True
5,2,4185,0.015760,0.016492,0.000732,True
1,4169,4304,0.013983,0.014928,0.000945,True
7,2889,4152,0.019932,0.020924,0.000992,True
21,22,3997,0.018880,0.019957,0.001077,True
2,7,4232,0.020780,0.022262,0.001482,True



=== weather_vector_v05_candidate vs baseline (largest regressions last) ===


,venue_id,sample_count_baseline,ece_overall_baseline,ece_overall_candidate,ece_delta,reliable_baseline
12,10,4110,0.014923,0.015425,0.000502,True
11,2395,4114,0.018695,0.019395,0.000700,True
1,4169,4304,0.013983,0.014764,0.000781,True
5,2,4185,0.015760,0.016599,0.000839,True
7,2889,4152,0.019932,0.020915,0.000983,True
21,22,3997,0.018880,0.019874,0.000994,True
2,7,4232,0.020780,0.021991,0.001211,True
30,5340,109,0.038569,0.040234,0.001665,True



=== geometry_plus_weather_v05_candidate vs baseline (largest regressions last) ===


,venue_id,sample_count_baseline,ece_overall_baseline,ece_overall_candidate,ece_delta,reliable_baseline
16,2681,4064,0.015813,0.021122,0.005310,True
5,2,4185,0.015760,0.021722,0.005962,True
28,4705,3824,0.015205,0.022822,0.007617,True
13,3,4103,0.016032,0.023857,0.007825,True
33,3949,32,0.095968,0.104077,0.008109,False
1,4169,4304,0.013983,0.022541,0.008558,True
30,5340,109,0.038569,0.069255,0.030685,True
32,2735,42,0.066350,0.125069,0.058719,False


## 14. Calibration by roof status

In [16]:
if comparison is not None:
    rows = []
    for variant in comparison:
        for label, s in comparison[variant]["weather_subgroups"].items():
            if label.startswith("roof_"):
                rows.append({"variant": variant, "roof_status": label, "sample_count": s["sample_count"], "ece": s["ece"], "log_loss": s["log_loss"]})
    roof_df = pd.DataFrame(rows)
    for status in roof_df["roof_status"].unique():
        print(f"\n--- {status} ---")
        display(roof_df[roof_df["roof_status"] == status].set_index("variant")[["sample_count", "ece", "log_loss"]])
else:
    print("Skipped -- no comparison available.")


--- roof_outdoor_open_air ---


,sample_count,ece,log_loss
variant,,,
selected_production_baseline,89736,0.014386,0.670052
weather_basic_v05_candidate,89736,0.014619,0.669578
weather_vector_v05_candidate,89736,0.014528,0.669636
geometry_plus_weather_v05_candidate,89736,0.016020,0.565370



--- roof_retractable_roof_open ---


,sample_count,ece,log_loss
variant,,,
selected_production_baseline,9402,0.016888,0.651449
weather_basic_v05_candidate,9402,0.016910,0.650906
weather_vector_v05_candidate,9402,0.016654,0.650876
geometry_plus_weather_v05_candidate,9402,0.014683,0.543829



--- roof_retractable_roof_closed ---


,sample_count,ece,log_loss
variant,,,
selected_production_baseline,19037,0.015493,0.682662
weather_basic_v05_candidate,19037,0.015724,0.682927
weather_vector_v05_candidate,19037,0.015468,0.682836
geometry_plus_weather_v05_candidate,19037,0.018244,0.561189



--- roof_fixed_indoor ---


,sample_count,ece,log_loss
variant,,,
selected_production_baseline,3957,0.019714,0.661906
weather_basic_v05_candidate,3957,0.019368,0.663217
weather_vector_v05_candidate,3957,0.018933,0.663122
geometry_plus_weather_v05_candidate,3957,0.017388,0.537248



--- roof_roof_status_unknown ---


,sample_count,ece,log_loss
variant,,,
selected_production_baseline,0,NaN,NaN
weather_basic_v05_candidate,0,NaN,NaN
weather_vector_v05_candidate,0,NaN,NaN
geometry_plus_weather_v05_candidate,0,NaN,NaN


## 15. Calibration by environmental subgroup (temperature, air density, wind strength)

In [17]:
if comparison is not None:
    rows = []
    for variant in comparison:
        for label, s in comparison[variant]["weather_subgroups"].items():
            if not label.startswith("roof_") and label not in ("complete_case_weather_available", "weather_unavailable"):
                rows.append({"variant": variant, "subgroup": label, "sample_count": s["sample_count"], "ece": s["ece"], "log_loss": s["log_loss"]})
    subgroup_df = pd.DataFrame(rows)
    for label in subgroup_df["subgroup"].unique():
        print(f"\n--- {label} ---")
        display(subgroup_df[subgroup_df["subgroup"] == label].set_index("variant")[["sample_count", "ece", "log_loss"]])
else:
    print("Skipped -- no comparison available.")


--- weather_quality_good ---


,sample_count,ece,log_loss
variant,,,
selected_production_baseline,120950,0.014430,0.670572
weather_basic_v05_candidate,120950,0.014524,0.670257
weather_vector_v05_candidate,120950,0.014410,0.670281
geometry_plus_weather_v05_candidate,120950,0.015955,0.561326



--- weather_quality_fair ---


,sample_count,ece,log_loss
variant,,,
selected_production_baseline,211,0.038210,0.627803
weather_basic_v05_candidate,211,0.042133,0.624118
weather_vector_v05_candidate,211,0.036415,0.618501
geometry_plus_weather_v05_candidate,211,0.045801,0.567885



--- high_temperature ---


...

,sample_count,ece,log_loss
variant,,,
selected_production_baseline,3940,0.019180,0.645655
weather_basic_v05_candidate,3940,0.017743,0.645685
weather_vector_v05_candidate,3940,0.017692,0.646295
geometry_plus_weather_v05_candidate,3940,0.018517,0.559579



--- venue_Oracle Park ---


,sample_count,ece,log_loss
variant,,,
selected_production_baseline,4114,0.018695,0.679693
weather_basic_v05_candidate,4114,0.018583,0.681220
weather_vector_v05_candidate,4114,0.019395,0.680871
geometry_plus_weather_v05_candidate,4114,0.021850,0.588200



--- near_wall_10ft ---


,sample_count,ece,log_loss
variant,,,
selected_production_baseline,3989,0.104664,1.345986
weather_basic_v05_candidate,3989,0.105345,1.341916
weather_vector_v05_candidate,3989,0.104971,1.341183
geometry_plus_weather_v05_candidate,3989,0.024209,1.004896


## 16. Paired-bootstrap intervals

Game_pk-level paired bootstrap: resamples GAMES with replacement so within-game correlation between plays is preserved; both the baseline and each candidate are evaluated on the SAME resampled rows each replicate.

In [18]:
if comparison is not None:
    training_eligible = df[df["eligible_for_training"].astype(bool)]
    val_df = training_eligible[training_eligible["season"].isin(VALIDATION_SEASONS)]
    val_df = _prepare_weather_columns(val_df)
    y_true = val_df["outcome_class"].astype(str)

    candidate_variants = [v for v in comparison if v != VARIANT_SELECTED_PRODUCTION_BASELINE]
    bootstrap_by_candidate = {}
    standardized_stable = {}
    for candidate in candidate_variants:
        bootstrap_by_candidate[candidate] = compute_paired_bootstrap(
            y_true, proba_by_variant[VARIANT_SELECTED_PRODUCTION_BASELINE], proba_by_variant[candidate], val_df["game_pk"],
        )
        standardized_stable[candidate] = check_standardized_predictions_stable(proba_by_variant[candidate])
        print(f"\n=== {candidate} vs {VARIANT_SELECTED_PRODUCTION_BASELINE} (n_reps={bootstrap_by_candidate[candidate]['log_loss']['n_reps']}, seed={bootstrap_by_candidate[candidate]['log_loss']['seed']}) ===")
        for metric, result in bootstrap_by_candidate[candidate].items():
            print(f"  {metric:15s} point_estimate={result['point_estimate']:+.6f}  95% CI=[{result['ci_low']:+.6f}, {result['ci_high']:+.6f}]")
else:
    bootstrap_by_candidate, standardized_stable = None, None
    print("Skipped -- no comparison available.")


=== weather_basic_v05_candidate vs selected_production_baseline (n_reps=500, seed=42) ===
  log_loss        point_estimate=-0.000307  95% CI=[-0.000525, -0.000109]
  ece             point_estimate=+0.000087  95% CI=[-0.000123, +0.000380]
  home_run_ece    point_estimate=+0.000694  95% CI=[+0.000277, +0.000986]
  brier_out       point_estimate=-0.000061  95% CI=[-0.000111, -0.000014]
  brier_single    point_estimate=+0.000019  95% CI=[+0.000001, +0.000037]
  brier_double    point_estimate=-0.000006  95% CI=[-0.000026, +0.000013]
  brier_triple    point_estimate=+0.000000  95% CI=[-0.000007, +0.000006]
  brier_home_run  point_estimate=-0.000084  95% CI=[-0.000151, -0.000024]



=== weather_vector_v05_candidate vs selected_production_baseline (n_reps=500, seed=42) ===
  log_loss        point_estimate=-0.000283  95% CI=[-0.000504, -0.000076]
  ece             point_estimate=-0.000022  95% CI=[-0.000255, +0.000228]
  home_run_ece    point_estimate=+0.000703  95% CI=[+0.000234, +0.000979]
  brier_out       point_estimate=-0.000048  95% CI=[-0.000100, -0.000002]
  brier_single    point_estimate=+0.000018  95% CI=[-0.000000, +0.000038]
  brier_double    point_estimate=-0.000005  95% CI=[-0.000029, +0.000017]
  brier_triple    point_estimate=-0.000001  95% CI=[-0.000008, +0.000006]
  brier_home_run  point_estimate=-0.000091  95% CI=[-0.000164, -0.000026]



=== geometry_plus_weather_v05_candidate vs selected_production_baseline (n_reps=500, seed=42) ===
  log_loss        point_estimate=-0.108173  95% CI=[-0.110618, -0.105166]
  ece             point_estimate=+0.001629  95% CI=[+0.000700, +0.002707]
  home_run_ece    point_estimate=-0.001660  95% CI=[-0.002444, -0.000861]
  brier_out       point_estimate=-0.020696  95% CI=[-0.021294, -0.020042]
  brier_single    point_estimate=-0.012240  95% CI=[-0.012700, -0.011798]
  brier_double    point_estimate=-0.011627  95% CI=[-0.012171, -0.011104]
  brier_triple    point_estimate=-0.000121  95% CI=[-0.000150, -0.000095]
  brier_home_run  point_estimate=-0.010489  95% CI=[-0.010990, -0.010078]


## 17. Actual-versus-standardized environment examples

In [19]:
if comparison is not None:
    best_weather_only = VARIANT_WEATHER_VECTOR_V05_CANDIDATE
    trained = trained_models[best_weather_only]
    feature_cols = trained.numeric_features + trained.categorical_features

    standardized_val_df = generate_standardized_environment_rows(val_df)
    actual_proba = predict_proba_ordered(trained, val_df[feature_cols])
    standardized_proba = predict_proba_ordered(trained, standardized_val_df[feature_cols])
    attribution = compute_weather_attribution(actual_proba, standardized_proba)
    attribution.index = val_df.index

    print(attribution["weather_run_value_effect"].describe())
    display(attribution.join(val_df[["venue_name", "outcome_class", "air_density_kg_m3", "following_wind_mps"]]).head(5))
else:
    attribution = None
    print("Skipped -- no comparison available.")

count    122132.000000
mean          0.001149
std           0.015992
min          -0.452961
25%          -0.001321
50%           0.000000
75%           0.002000
max           0.446242
Name: weather_run_value_effect, dtype: float64


,expected_run_value_actual_environment,expected_run_value_standard_environment,weather_run_value_effect,venue_name,outcome_class,air_density_kg_m3,following_wind_mps
370193,-0.190143,-0.188097,-0.002046,Nationals Park,out,1.188952,3.573140
370194,-0.242502,-0.242383,-0.000119,Nationals Park,out,1.188952,0.587754
370195,-0.128896,-0.130422,0.001527,Nationals Park,out,1.188952,3.041932
370196,0.072894,0.052825,0.020069,Nationals Park,out,1.188952,3.164333
370197,-0.083668,-0.079127,-0.004541,Nationals Park,out,1.188952,3.410168


## 18. Plays with largest weather effects

In [20]:
if attribution is not None:
    top = attribution["weather_run_value_effect"].abs().sort_values(ascending=False).head(8)
    detail = attribution.join(val_df[["venue_name", "outcome_class", "air_density_kg_m3", "following_wind_mps", "temperature_c"]])
    display(detail.loc[top.index])
else:
    print("Skipped -- no attribution available.")

,expected_run_value_actual_environment,expected_run_value_standard_environment,weather_run_value_effect,venue_name,outcome_class,air_density_kg_m3,following_wind_mps,temperature_c
461122,0.421317,0.874278,-0.452961,Coors Field,double,1.013328,1.864313,7.777778
440161,1.120073,0.673831,0.446242,Angel Stadium,home_run,1.184627,-1.251716,21.666667
460994,0.564628,0.941339,-0.376711,Coors Field,out,1.018024,-2.907553,13.333333
460844,0.577020,0.952839,-0.375819,Coors Field,out,1.066637,-1.335503,0.555556
460076,0.446319,0.819524,-0.373205,Coors Field,out,0.982155,0.399825,20.000000
460138,0.419624,0.787790,-0.368166,Coors Field,home_run,0.994750,1.418639,18.888889
460125,0.401631,0.766605,-0.364974,Coors Field,out,0.994750,0.239388,18.888889
460043,0.500288,0.863894,-0.363606,Coors Field,home_run,1.014923,-1.553594,11.111111


## 19. Physically plausible interpretation

Read the largest-effect plays above together with `air_density_kg_m3` and `following_wind_mps`: a physically plausible weather effect looks like `weather_run_value_effect` being POSITIVE (favorable to the batter) when air density is LOW (thin air carries fly balls farther -- the classic Coors Field effect) or `following_wind_mps` is strongly positive (tailwind), and NEGATIVE when air density is high or the wind is a strong headwind. This mirrors the Version 0.4 park-geometry qualitative check and the well-documented real-world Coors Field effect (see `mlb_luck_score.data.weather_physics` module docstring's worked example: Coors Field's estimated air density is roughly 15-20% below a sea-level reference at the same temperature/humidity). This check is qualitative, not automated -- see the adoption recommendation below for why "physically plausible effects" is never auto-passed.

## 20. Adoption recommendation

In [21]:
if comparison is not None:
    recommendation = recommend_weather_adoption(
        comparison, bootstrap_by_candidate, candidate_variants=tuple(candidate_variants),
        standardized_predictions_stable=standardized_stable,
    )
    import json
    print(json.dumps(recommendation, indent=2, default=str))
else:
    recommendation = None
    print("Skipped -- no comparison available.")

{
  "per_candidate": {
    "weather_basic_v05_candidate": {
      "improves_log_loss": true,
      "log_loss_delta": -0.00030654133688667873,
      "bootstrap_supports_improvement_or_no_meaningful_regression": true,
      "log_loss_bootstrap": {
        "metric": "log_loss",
        "point_estimate": -0.00030654133688667873,
        "ci_low": -0.0005250645554794763,
        "ci_high": -0.0001094387948064546,
        "confidence_level": 0.95,
        "n_reps": 500,
        "seed": 42,
        "resampling_unit": "game_pk"
      },
      "ece_not_materially_worse": true,
      "ece_delta": 8.736213044164241e-05,
      "home_run_ece_not_materially_worse": true,
      "home_run_ece_delta": 0.0006942817397936093,
      "no_material_venue_regressions": true,
      "material_venue_regressions": [],
      "no_material_subgroup_regressions": true,
      "material_subgroup_regressions": [],
      "sufficient_weather_coverage": true,
      "weather_coverage_rate": 0.8042036485114467,
      "no_evi

## 21. Explicit limitations

- **Venue-environment reference data is agent-researched, not human-reviewed.** Coordinates are approximate (city-block precision); every record's `review_status` is `agent_sourced_pending_human_review`.
- **Only 30 primary venues have weather station assignments.** Temporary/neutral-site venues (Field of Dreams, London/Mexico City Series, Little League Classic, Rickwood Field, the 2021 Blue Jays alternate homes) have NO weather -- rows are preserved with `weather_status="venue_environment_unavailable"`, never fabricated.
- **Humidity/pressure come from a station up to ~23km from the venue** (see section 2), not an on-site sensor -- a documented approximation, not a measurement at home plate.
- **Indoor/closed-roof games have NO effective weather.** No reviewed indoor-climate assumption exists in this repository -- `effective_temperature_c`/`humidity`/`pressure`/`air_density_kg_m3` are left null for `fixed_indoor`/`retractable_roof_closed` games (roughly 18% of plays), though wind is correctly forced to zero (a physical certainty, not a guess).
- **Stadium true-north field orientation was never independently sourced** -- this repository relies entirely on MLB's own pre-rotated, park-relative wind text (`"Out To CF"`, `"L To R"`, etc.), which is what makes the following/head/crosswind decomposition possible without that data; it is not an independent verification of MLB's own rotation.
- **The standardized-environment counterfactual is a fixed, documented reference (ISA sea level, zero wind, roof-neutral), not a claim that this is "typical" game weather** -- it isolates the weather-specific effect, holding every non-weather feature constant.
- **`weather_run_value_effect` is a separate decomposition, not additive luck.** Never sum it with `raw_contact_luck_runs` -- see `mlb_luck_score.scoring.weather_attribution` module docstring for why that would double-count weather.
- **Contact Luck v0.5 remains a contact-only research prototype.** It does not yet include exact defensive positioning, defensive execution, batter-runner advancement, or full multi-factor Shapley attribution.
- **2025 was never touched** by this analysis.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.